In [ ]:
from pathlib import Path
import ixmp4
import pyam
import nomenclature

# Import scenario data and definitions (a.k.a "Project Template")

The **Scenario Compass** and other ongoing model comparison projects use the **[common-definitions](https://github.com/iamconsortium/common-definitions)** codelists and mappings.
You can work with these definitions using the **[nomenclature](https://nomenclature-iamc.readthedocs.io)** package or you can download the list of variables in xlsx format from https://files.ece.iiasa.ac.at/common-definitions/common-definitions-template.xlsx.

In [ ]:
df = pyam.IamDataFrame(<file>)

In [ ]:
definition = nomenclature.DataStructureDefinition("common-definitions/definitions/")

You can use the `rename()` method to update model, scenario and region names...

Please follow these guidelines:
- The **model** name should have whitespace (no underscore) and include the version number
- The **scenario** name should refer clearly to the project
- The **region** names must be in line with the model-registration 

```python
df.rename(model={"Previous Name": "New Name"}, inplace=True}
```

# Rename variables and units

The **common-definitions codelists** include attributes for legacy projects including **NAVIGATE** and **ENGAGE**.

The next cell creates a mapping of legacy-variable-names to their **common-definitions** equivalent and applies it to the scenario data.

In [ ]:
project = ["navigate", "engage"]
legacy_mapping = {}

for _project in project:
    for code, attrs in definition.variable.items():
        if _project in attrs.extra_attributes:
            legacy_mapping[attrs.__getattr__(_project)] = code
    
    df.rename(variable=legacy_mapping, inplace=True)

The **common-definitions** use naming conventions for units consistent with the **[iam-units](https://github.com/IAMconsortium/units)** package.

The next cell harmonizes to the new unit convention where possible.

In [ ]:
df.rename(
    unit={
        "US$2010/kW OR local currency/kW": "USD_2010/kW",
        "US$2010/kW": "USD_2010/kW",
        "billion US$2010/yr": "billion USD_2010/yr",
        "billion US$2010/yr OR local currency/yr": "billion USD_2010/yr",
        "billion US$2010/yr or local currency/yr": "billion USD_2010/yr",
        "US$2010/t CO2": "USD_2010/t CO2",
        "US$2010/tCO2": "USD_2010/t CO2",
        "US$2010/t CO2 or local currency/t CO2": "USD_2010/t CO2",
        "million Ha/yr": "million ha",
        "Million": "million",
        "Mt NOx/yr": "Mt NO2/yr",  
        "Mt N2O/yr": "kt N2O/yr",
        "US$2010/GJ": "USD_2010/GJ",
        "bn m2": "billion m2",
        "bn tkm/yr": "billion tkm/yr",
        "bn pkm/yr": "billion pkm/yr",
    },
    inplace=True,
)

There are some known renamings that have not yet been added to **common-definitions**. As a quick-fix, this can be done directly in this notebook.

In [ ]:
variable_mapping = {
    "Agricultural Demand|Crops|Energy": "Agricultural Demand|Crops|Bioenergy",
    "Agricultural Demand|Crops|Energy|1st generation": "Agricultural Demand|Crops|Bioenergy|1st Generation",
    "Agricultural Demand|Crops|Energy|2nd generation": "Agricultural Demand|Crops|Bioenergy|2nd Generation",
    "Carbon Sequestration|CCS|Biomass": "Carbon Capture|Geological Storage|Biomass",
    "Carbon Sequestration|CCS|Biomass|Energy|Supply": "Carbon Capture|Energy|Supply|Biomass",
    "Carbon Sequestration|CCS|Fossil": "Carbon Capture|Energy|Fossil",
    "Carbon Sequestration|CCS|Fossil|Energy|Demand|Industry": "Carbon Capture|Energy|Demand|Industry",
    "Carbon Sequestration|CCS|Fossil|Energy|Supply": "Carbon Capture|Energy|Supply|Fossil",
    "Carbon Sequestration|CCS|Industrial Processes": "Carbon Capture|Industrial Processes",
    "Carbon Sequestration|Land Use|Afforestation": "Carbon Removal|Land Use|Re/Afforestation",
    "Carbon Sequestration|Direct Air Capture": "Carbon Removal|Geological Storage|Direct Air Capture",
    "Carbon Sequestration|Enhanced Weathering": "Carbon Removal|Enhanced Weathering",
    "Carbon Sequestration|Land Use": "Carbon Removal|Land Use",
    "Yield|Cereal": "Yield|Cropland|Cereals",
    "Yield|Oilcrops": "Yield|Cropland|Oil Crops",
    "Yield|Sugarcrops": "Yield|Cropland|Sugar Crops",
}

df.rename(variable=variable_mapping, inplace=True)

You can remove variables that are not included in **common-definitions** and/or that are not relevant to the analysis.

In [ ]:
df.filter(
    variable=[
        "*AR6 climate diagnostics*",
    ],
    keep=False,
    inplace=True
)

As a final step, you can validate that the remaining variables and units in the scenario data are consistent with **common-definitons**.

The next cell will show any inconsistent or unexpected variables and units in the scenario data.

In [ ]:
definition.validate(df, dimensions=["variable"])

# Data validation

Some variables in **common-definitions** have obvious bounds or ranges (e.g., non-negative timeseries data, percentages-point indicators between 0 and 100).

The next cell creates a criteria-list of the validation items and creates a **DataValidator** instance. The following cell runs the scenario data against the validation criteria and shows any violations.

In [ ]:
validation_args = ["upper_bound", "lower_bound", "value", "rtol", "atol", "range"]

validation_list = list()

for name, variable in definition.variable.items():
    if any([i in validation_args for i in variable.extra_attributes]):
        validation_list.append(
            dict(
                variable=name,
                validation=[dict([(key, value) for key, value in variable.extra_attributes.items() if key in validation_args])]
            )
        )

validator = nomenclature.processor.DataValidator(criteria_items=validation_list, file=".")

In [ ]:
validator.apply(df)

# Meta indicators

You can set meta indicators for the scenario data so that users can easily cross-reference and cite the data.

In [ ]:
df.set_meta("<project name>", "Project")
df.set_meta("Author et al., 2015", "Scientific Manuscript (Citation)")
df.set_meta("10.1234/1234.567", "Scientific Manuscript (DOI)")
df.set_meta("0.1234/1234.567", "Data Source (DOI)")

# Data export

In [ ]:
df.to_excel("<file>.xlsx")